# DepthWizard v4 — training (Modal Notebook, **1× H100**)

Every cell shells out to `V4_Kaggle/train.py`. Its DDP machinery short-circuits
at `world_size == 1`, so this is a plain `python train.py` — no `torchrun`, no
NCCL env. Flag rationale lives in
[`V4_Kaggle/README.md` §5.2](../V4_Kaggle/README.md).

## Before you run a single cell

| Sidebar tab | Set to |
|---|---|
| Compute profile | **GPU H100, 12 CPU, 48 GiB** — `num_workers=12` needs those cores |
| Compute profile | **Idle timeout to the maximum** |
| Filesystem | Volume `depthwizard-data` → `/mnt/depthwizard-data` (from `01_data_prep`) |
| Filesystem | Volume `depthwizard-results` → `/mnt/depthwizard-results` |
| Secrets | `dw-hf` (`HF_TOKEN`) — `DINOv3-SAT` is gated |

Accept the licence for `facebook/dinov3-vitl16-pretrain-sat493m` first.

**Run the cells in order and do not skip one.** They are ordered by cost: the
setup and sanity cells are minutes, the smoke run is ~10 min, the real run is
~3 h at **~$4.90/h** all-in (H100 + 12 cores + 48 GiB) — call it **~$14**.

**The kernel is ephemeral; the Volumes are not.** Checkpoints go straight to
`/mnt/depthwizard-results/v4m`, so a kernel that dies mid-run is resumable from
`last_full.pt` (cell 10) rather than a $14 retrain. The training cell blocks
while it runs, which is what keeps the kernel from being idle — but raise the
idle timeout anyway, because the moment training *ends* the clock starts.


In [ ]:
# 1. Code + deps.  requirements.txt omits torch on purpose ("the host's CUDA
#    build is kept") and a Modal container has no host build, so pin it here.
#    If that wheel is gone, bump the version and re-run the sanity cell.
!git clone --depth 1 https://github.com/AbhayKale332/DepthWizard.git /root/DepthWizard 2>/dev/null || echo "already cloned"
%cd /root/DepthWizard/FineTunning/V4_Kaggle
!apt-get -qq update && apt-get -qq install -y rsync libglib2.0-0
%uv pip install torch==2.10.0 --extra-index-url https://download.pytorch.org/whl/cu128
%uv pip install -r requirements.txt
# If the %uv magic rejects a flag, the shell form works too:
#   !uv pip install --system -r requirements.txt

In [ ]:
import json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

CODE    = "/root/DepthWizard/FineTunning/V4_Kaggle"
DATA    = "/mnt/depthwizard-data"      # Volume, attached in the sidebar
SCRATCH = "/scratch/dwdata"            # container-local; dies with the kernel


def sh(cmd, cwd=CODE):
    """Run and fail loudly. Never build cmd from anything but the literals here."""
    print(f"\n[dw] $ {shlex.join(cmd)}", flush=True)
    t = time.time()
    if subprocess.run(cmd, cwd=cwd, check=False).returncode:
        raise RuntimeError(f"[dw] failed after {(time.time() - t) / 60:.1f} min: {cmd[0]}")
    print(f"[dw] ok ({(time.time() - t) / 60:.1f} min)", flush=True)


def stores(root):
    """Print every store's tile count. Read these.

    v3 wrote a gamus/val holding 129 of 400 tiles and still reported success,
    because a tile whose download failed was swallowed as `skip: ...`.
    """
    n = 0
    print(f"\n[dw] stores under {root}:")
    for idx in sorted(Path(root).glob("*/*/index.json")):
        n += 1
        d = json.loads(idx.read_text())
        # .parents[1], not .parent — .parent is the split. That is the ba78dd8 bug.
        name = f"{idx.parents[1].name}/{idx.parent.name}"
        gib = sum(f.stat().st_size for f in idx.parent.glob("*.npy")) / 2**30
        ok = "bounds" if (idx.parent / "stretch_bounds_2_98.npy").exists() else "NO-BOUNDS"
        print(f"    {name:<20} n={d['n']:>6}  tile={d['tile_px']:>4}  "
              f"gsd={d['gsd_m']:<5}  seg={'yes' if d.get('has_seg') else 'no':<3}  "
              f"{gib:6.2f} GiB  {ok}")
    return n


RESULTS = "/mnt/depthwizard-results"
OUT     = f"{RESULTS}/v4m"

for p, why in ((DATA, "01_data_prep.ipynb's output"), (RESULTS, "where best.pt lands")):
    assert Path(p).is_dir(), f"{p} is not mounted ({why}) — sidebar → Filesystem"
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set — sidebar → Secrets → 'dw-hf'"

# Cache the encoder on the results Volume, not the container: otherwise every
# kernel restart re-downloads ~1.2 GiB of gated weights.
os.environ["HF_HOME"] = f"{RESULTS}/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

Path(OUT).mkdir(parents=True, exist_ok=True)
print(f"[dw] outputs -> {OUT}")
print(f"[dw] /scratch free: {shutil.disk_usage('/scratch').free / 2**30:.0f} GiB "
      f"(staging needs ~36)")

In [ ]:
# 3. Pull the gated encoder once, and prove the hardware is what you asked for.
#
# Doing this here rather than inside train.py makes a bad token a 30-second
# error instead of a late, illegible one — train.py:314-315 only *warns* about a
# missing token.
ENCODER = "facebook/dinov3-vitl16-pretrain-sat493m"

from huggingface_hub import snapshot_download
snapshot_download(ENCODER, token=os.environ["HF_TOKEN"])
try:
    from transformers import AutoImageProcessor
    AutoImageProcessor.from_pretrained(ENCODER, token=os.environ["HF_TOKEN"])
except Exception as e:                    # noqa: BLE001
    print(f"[dw] processor not cached ({e}); preprocess.py's constant fallback applies")

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  gpu{i}: {p.name} {p.total_memory / 2**30:.1f}GiB sm_{p.major}{p.minor}"
          f"{'' if p.major >= 8 else '  NO bf16 -> set amp_dtype fp16 below'}")
import cv2, rasterio, h5py, tifffile, transformers                    # noqa: F401
print(f"imports ok | /dev/shm {shutil.disk_usage('/dev/shm').total / 2**30:.1f} GiB "
      f"| nproc {os.cpu_count()}")

In [ ]:
# 4. The profile. Every key is a config.py dataclass field; parse_config
#    auto-generates one --<field> per field, so this dict IS the command line.
#
# Two that are specific to this port:
#   batch_size 24     — config.py:184-190 measured 32 -> 74 GB then OOM at e7.
#   session_minutes 0 — that second cap firing before the other two truncated
#                       v4-2's cosine at 81% and cost 0.36 m. One notebook
#                       kernel has no 12 h Kaggle session to survive.
FLAGS = {
    "data_root": SCRATCH, "output_dir": OUT,
    # gamus MUST stay first: loaders.py:31-43 selects best.pt on the first
    # source that has a val split.
    "datasets": "gamus,synrs3d_g1,synrs3d_g05,dfc23_g050",
    "amp_dtype": "bf16",                 # Hopper: disables the GradScaler path entirely
    "grad_checkpoint_encoder": "false", "grad_checkpoint_decoder": "false",
    "batch_size": "24", "grad_accum": "1", "eval_batch_mult": "2",
    "num_workers": "12", "prefetch_factor": "6", "compile_model": "false",
    "encoder_unfreeze_blocks": "0", "llrd": "0.90",
    "epochs": "30", "eval_every": "1",
    "max_minutes": "150", "session_minutes": "0",
    "save_full_state": "true", "full_state_every": "5",
    "make_zip": "false",                 # build_zip duplicates ~4 GB onto a billed volume
}

CROPS_PER_EPOCH = 12000                  # config.py:91
IMG_PER_S = 45.0                         # v3 on an H100; replace from the 3-epoch calibration
RATE = 3.95 + 12 * 0.0472 + 48 * 0.0080  # $/h: H100 + 12 cores + 48 GiB


def cost(epochs):
    """(hours, dollars) for one run. +0.7 h for the final sliding+TTA eval."""
    h = epochs * CROPS_PER_EPOCH / IMG_PER_S / 3600 + 0.7
    return h, h * RATE


def argv(**ov):
    """The exact argv train.py sees. `extra` is appended raw and wins
    (argparse takes the last occurrence)."""
    extra = ov.pop("extra", "")
    f = FLAGS | {k: str(v) for k, v in ov.items() if v not in ("", 0, None)}
    return [x for k, v in f.items() for x in (f"--{k}", v)] + shlex.split(extra)


# config.py's parser ends in parse_known_args, so a flag that is NOT a real
# dataclass field is SILENTLY DROPPED and the run proceeds on the default you
# thought you overrode. A whole GPU-hour for a typo. This is that guard.
from dataclasses import fields
sys.path.insert(0, CODE)
from config import Config
unknown = sorted((set(FLAGS) | {"smoke", "freeze_epochs", "resume"})
                 - {f.name for f in fields(Config)})
if unknown:
    raise RuntimeError(f"[dw] not Config fields, would be silently dropped: {unknown}")
print(f"[dw] {len(FLAGS) + 3} flags all exist in Config")

print("\npython train.py " + shlex.join(argv()))
print(f"\n{'epochs':>7} {'hours':>7} {'$':>8}   (at {IMG_PER_S} img/s)")
for e in (16, 26, 30, 40):
    h, d = cost(e)
    print(f"{e:>7} {h:>7.2f} {d:>8.2f}{'  <- default' if e == 30 else ''}")

In [ ]:
# 5. The 166 offline tests. ~2 min of H100 time to rule out a broken checkout.
sh([sys.executable, "-m", "pytest", "-q"])

In [ ]:
# 6. Stage the data off the Volume. NOT an optimisation.
#
# The trainer opens every shard mmap_mode="r" (packed.py:141,198) and reads
# random ~0.79 MB windows ~45/s over ~36 GiB. Served off a FUSE-backed Volume
# that is what Logs/v3/gpu_disk_guard.log calls STARVED. ~10 min, once per kernel.
sh(["rsync", "-a", "--info=progress2", f"{DATA}/", f"{SCRATCH}/"], cwd="/")
if not stores(SCRATCH):
    raise RuntimeError(f"[dw] nothing staged from {DATA} — run 01_data_prep.ipynb first")

In [ ]:
# 7. The runner. `python -u` because a block-buffered pipe would stall the cell
#    output for minutes; tee keeps a log on the Volume so a dropped browser
#    loses nothing.
def run(args, log):
    Path(log).parent.mkdir(parents=True, exist_ok=True)
    cmd = (f"{shlex.quote(sys.executable)} -u train.py {shlex.join(args)} "
           f"2>&1 | tee -a {shlex.quote(log)}")
    print(f"[dw] $ {cmd}\n", flush=True)
    t = time.time()
    rc = subprocess.run(["bash", "-c", cmd], cwd=CODE, check=False).returncode
    print(f"\n[dw] {'ok' if not rc else 'FAILED'} after {(time.time() - t) / 60:.1f} min")
    if rc:
        raise RuntimeError(f"[dw] train.py exited {rc} — see {log}")


# 8. Smoke: 24 crops, 2 epochs, batch 2 (config.py:313-334). ~10 min, ~$1.
#    Do not skip it.
run(argv(output_dir=f"{RESULTS}/smoke_v4m", extra="--smoke"),
    f"{RESULTS}/smoke_v4m/run.log")

In [ ]:
# 9. The real run. ~3 h, ~$14.
#
# max_minutes SIZES the LR cosine (train.py:626-632). Moving EPOCHS without it
# is how v4-2 annealed to 5.09e-05 instead of 8.32e-06 — so it is re-derived
# here, never set by hand.
EPOCHS = 30        # set to 3 with a short max_minutes to calibrate IMG_PER_S first

ov = {} if EPOCHS == int(FLAGS["epochs"]) else {
    "epochs": EPOCHS, "max_minutes": round(cost(EPOCHS)[0] * 60 * 0.9)}
h, d = cost(EPOCHS)
print(f"[dw] projected {h:.2f} h, ${d:.2f} at {IMG_PER_S} img/s")

run(argv(**ov), f"{OUT}/run.log")

In [ ]:
# 10. RECOVERY — only if the kernel died mid-run. Re-run cells 1-3, 6, 7 first.
#
# last_full.pt carries the optimiser, scaler, EMA, teacher, RNG and elapsed
# minutes (full_state_every=5). The log should show the LR continuing *down* the
# cosine, not re-warming.
#
# run(argv(resume=f"{OUT}/last_full.pt"), f"{OUT}/run.log")

# 11. RECOVERY — training finished but the exports did not. ~$2, not ~$14.
# train.py:938-948 reads history/best back out of the committed metrics.json and
# runs final TTA, sliding eval, figures and ONNX.
#
# run(argv(epochs="0", freeze_epochs="0"), f"{OUT}/finalize.log")

In [ ]:
# 12. What survived. best.pt alone is useless — preproc.json is the
#     preprocessing contract infer/engine.py reads back.
for f in ("best.pt", "metrics.json", "config.json", "preproc.json", "run.log"):
    p = Path(OUT) / f
    print(f"  {f:<16} {'%.1f MiB' % (p.stat().st_size / 2**20) if p.exists() else 'MISSING'}")

m = Path(OUT) / "metrics.json"
if m.exists():
    d = json.loads(m.read_text())
    print("\n[dw] best:", json.dumps(d.get("best", d), indent=2)[:600])

## Watch these in the log while it runs

- **eval RMSE changing between epochs** — byte-identical lines are a dead epoch
- skipped-gradient fraction ~0 — bf16 disables the scaler, so anything else
  means `--amp_dtype` did not take
- `nrm=` never `nan`
- `vram=` flat near 58 G, not creeping
- `[sched]` at startup saying the cosine reaches 1.0
- `img/s` near 45 — below that the loader is starving, and cell 6 did not stage

## Getting the weights out

From the notebook: the Filesystem panel under `depthwizard-results/v4m`.
From your machine:

```bash
for f in best.pt metrics.json config.json preproc.json; do
  modal volume get depthwizard-results v4m/$f .; done
```

## Then stop the kernel

An H100 bills while it idles. Sidebar → **Stop kernel** the moment cell 12 looks
right. `/scratch` goes with it; both Volumes do not.
